# Importação Pescadores RGP

Leitura de TODOS os arquivos Excel de `/Volumes/pescador/staging/dados_rgp/` (26 UFs: AC, AL, AM, AP, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ, RN, RO, RR, RS, SC, SE, SP, TO).

**Camada Bronze:** Cada UF é salva como uma tabela separada em `pescador.layer_bronze` (ex: `pescadores_AC`, `pescadores_AL`, ...).

**Camada Bronze:** Consolidação em uma única tabela `pescador.layer_bronze.pescadores_BR` com coluna `UF` como primeiro campo, identificando a origem dos dados. Inclui o campo adicional `CPF_PRIMEIRO_NOME` que concatena a parte numérica do CPF com o primeiro nome do pescador.

In [0]:
%pip install openpyxl

import re
import pandas as pd
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import lit, regexp_replace, split, concat, col

def sanitize_col(name):
    name = name.strip()
    name = name.replace("ç", "c").replace("ã", "a").replace("á", "a")
    name = name.replace("é", "e").replace("í", "i").replace("ó", "o")
    name = name.replace("ú", "u").replace("â", "a").replace("ê", "e")
    name = name.replace("ô", "o").replace("û", "u")
    name = re.sub(r"[^A-Za-z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

In [0]:
# Lista dinamicamente todos os arquivos Excel do diretório
arquivos_dir = "/Volumes/pescador/staging/dados_rgp/"
arquivos = []

for f in dbutils.fs.ls(arquivos_dir):
    name = f.name
    # Filtra apenas arquivos .xlsx (ignora pastas como "Dados/")
    if name.endswith(".xlsx"):
        uf = name.split(" ")[0]  # Extrai UF das primeiras letras antes do espaço
        arquivos.append((uf, arquivos_dir + name))

# Ordena alfabeticamente por UF
arquivos.sort(key=lambda x: x[0])

print(f"Encontrados {len(arquivos)} arquivos Excel:")
for uf, path in arquivos:
    print(f"  {uf} -> {path}")

dfs_spark = []

for uf, path in arquivos:
    print(f"\nLendo: {path} (UF={uf})")
    df_pd = pd.read_excel(path)

    # Sanitiza nomes de colunas no pandas (sem acessar .columns do Spark no loop)
    new_cols = [sanitize_col(c) for c in list(df_pd.columns)]
    df_pd.columns = new_cols

    df_spark = spark.createDataFrame(df_pd)

    # Renomeia coluna UF existente para evitar conflito
    if "UF" in new_cols:
        df_spark = df_spark.withColumnRenamed("UF", "UF_ESTADO")
        new_cols = [c if c != "UF" else "UF_ESTADO" for c in new_cols]

    # Adiciona coluna UF (derivada do nome do arquivo) como primeiro campo via select
    df_spark = df_spark.select(lit(uf).alias("UF"), *new_cols)

    # Salva tabela parcial na camada bronze (uma por UF)
    tabela_bronze = f"pescador.layer_bronze.pescadores_{uf}"
    df_spark.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(tabela_bronze)
    print(f"  Tabela bronze criada: {tabela_bronze}")

    dfs_spark.append(df_spark)
    n_reg = df_spark.count()
    print(f"  Registros: {n_reg:,} | Colunas: {len(new_cols) + 1}")

# Unifica todos os DataFrames Spark
df_unified = reduce(DataFrame.unionByName, dfs_spark)

# Adiciona campo CPF_PRIMEIRO_NOME: parte numérica do CPF + primeiro nome
df_unified = df_unified.withColumn(
    "CPF_PRIMEIRO_NOME",
    concat(
        regexp_replace(col("CPF"), "[^0-9]", ""),
        lit("_"),
        split(col("Nome_do_Pescador"), " ").getItem(0)
    )
)

# Cria/sobrescreve a tabela consolidada
df_unified.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pescador.layer_bronze.pescadores_BR")

print("\nTabela pescador.layer_bronze.pescadores_BR criada/sobrescrita com sucesso!")

In [0]:
# Resumo da tabela consolidada
total_registros = spark.table("pescador.layer_bronze.pescadores_BR").count()
total_colunas = len(spark.table("pescador.layer_bronze.pescadores_BR").columns)

print("=" * 50)
print(f"Tabela: pescador.layer_bronze.pescadores_BR")
print(f"Total de registros: {total_registros:,}")
print(f"Total de colunas: {total_colunas}")
print("=" * 50)

# Resumo por UF
print("\nRegistros por UF:")
display(spark.table("pescador.layer_bronze.pescadores_BR").groupBy("UF").count().orderBy("UF"))

# Schema da tabela
print("\nSchema:")
spark.table("pescador.layer_bronze.pescadores_BR").printSchema()

# Amostra dos dados
display(spark.table("pescador.layer_bronze.pescadores_BR").limit(10))

# Criação da tabela Silver PESCADORES (deduplicada)

Cria a tabela `pescador.layer_silver.pescadores` a partir de `pescador.layer_bronze.pescadores_BR`, removendo registros com `CPF_PRIMEIRO_NOME` duplicado. 

In [0]:
# Cria a tabela silver pescadores a partir do bronze, eliminando registros com CPF_PRIMEIRO_NOME duplicado
# Se um CPF aparece mais de uma vez, TODAS as ocorrencias sao removidas
df_bronze = spark.table("pescador.layer_bronze.pescadores_BR")

# Conta quantas vezes cada CPF_PRIMEIRO_NOME aparece
from pyspark.sql.functions import col, count as count_fn

df_counts = df_bronze.groupBy("CPF_PRIMEIRO_NOME").agg(
    count_fn("*").alias("qtd_ocorrencias")
)

# Filtra apenas CPFs que aparecem uma unica vez (nao duplicados)
cpfs_unicos = df_counts.filter(col("qtd_ocorrencias") == 1).select("CPF_PRIMEIRO_NOME")

# Join: mantem apenas os registros cujo CPF aparece uma unica vez
df_silver = df_bronze.join(
    cpfs_unicos,
    on="CPF_PRIMEIRO_NOME",
    how="inner"
).select(
    "CPF_PRIMEIRO_NOME",
    "UF",
    "Municipio",
    "Categoria",
    "Forma_de_Atuacao",
)

df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("pescador.layer_silver.pescadores")

print("Tabela pescador.layer_silver.pescadores criada/sobrescrita com sucesso!")
print(f"Registros duplicados completamente removidos (nenhuma ocorrencia mantida)")

In [0]:
# Resumo da tabela silver criada
total_bronze = spark.table("pescador.layer_bronze.pescadores_BR").count()
total_silver = spark.table("pescador.layer_silver.pescadores").count()
removidos = total_bronze - total_silver

total_colunas = len(spark.table("pescador.layer_silver.pescadores").columns)

print("=" * 60)
print(f"Tabela: pescador.layer_silver.pescadores")
print(f"Total de registros: {total_silver:,}")
print(f"Total de colunas: {total_colunas}")
print(f"Registros removidos (duplicados): {removidos:,}")
print(f"Bronze: {total_bronze:,} -> Silver: {total_silver:,}")
print("=" * 60)

# Schema da tabela
print("\nSchema:")
spark.table("pescador.layer_silver.pescadores").printSchema()

# Amostra dos dados
display(spark.table("pescador.layer_silver.pescadores").limit(10))

In [0]:
%sql

select count(*) from pescador.layer_silver.pescadores